In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

## 반복측정 ANOVA

## <문제1>

한 제약회사에서 새로운 항불안 약물의 효과를 검증하고자 합니다. 20명의 참가자에게 3가지 다른 용량(저용량, 중용량, 고용량)의 약물을 각각 1주일씩 투여하고, 각 주말에 불안 수준을 측정했습니다. (점수가 낮을수록 불안 수준이 낮음을 의미).

- 불안 수준 점수 (1점 ~ 100점)

In [2]:
file_path = "./data/repeated_anova.csv"
df = pd.read_csv(file_path)
df

,참가자ID,저용량,중용량,고용량
0,1,75,68,61
1,2,78,71,64
2,3,72,65,58
3,4,77,70,63
4,5,80,73,66
5,6,74,67,60
6,7,76,69,62
7,8,73,66,59
8,9,79,72,65
9,10,71,64,57


In [3]:
### 그룹 나누기
gLow = df["저용량"].to_numpy()
gMid = df["중용량"].to_numpy()
gHigh = df["고용량"].to_numpy()

In [4]:
### 정규성 검정 (gLow)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gLow)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.785484790802002
귀무가설 채택


In [5]:
### 정규성 검정 (gMid)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gMid)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.7854843735694885
귀무가설 채택


In [6]:
### 정규성 검정 (gHigh)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gHigh)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.7854846715927124
귀무가설 채택


In [7]:
### wide 데이터 long 데이터로 변경
df_melt = pd.melt(df,
                  id_vars="참가자ID",
                  value_vars=["저용량", "중용량", "고용량"],
                  var_name="용량",
                  value_name="불안점수"
                  )
df_melt

,참가자ID,용량,불안점수
0,1,저용량,75
1,2,저용량,78
2,3,저용량,72
3,4,저용량,77
4,5,저용량,80
5,6,저용량,74
6,7,저용량,76
7,8,저용량,73
8,9,저용량,79
9,10,저용량,71


In [8]:
### 구형성 검정
import pingouin as pg

# H0 : 데이터는 구형성을 만족한다.
# H1 : 데이터는 구형성을 만족하지 않는다.

result, _, _, _, p = pg.sphericity(df_melt,            ## 원본데이터
                                   dv="불안점수",      # 종속변수 컬럼명
                                   within="용량",      # 독립변수 컬럼명
                                   subject="참가자ID"  # 고유식별자 컬럼명
                                   )

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")
print("유의수준 5%하에서 귀무가설을 채택한다. 즉, 데이터는 구형성을 만족한다.")

검정통계량의 p-value : 1.0
귀무가설 채택
유의수준 5%하에서 귀무가설을 채택한다. 즉, 데이터는 구형성을 만족한다.


In [9]:
### 반복측정 ANOVA
import pingouin as pg

# H0 : 용량별 불안점수 평균이 같다.
# H1 : 적어도 하나의 용량은 불안점수 평균이 다르다.

result = pg.rm_anova(df_melt,                ## 원본 데이터
                     dv="불안점수",           # 종속변수 컬럼명
                     within="용량",           # 독립변수 컬럼명 
                     subject="참가자ID"       # 고유식별자 컬럼명
                     )         

p = result.loc[0,"p-unc"]

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.0
귀무가설 기각


In [10]:
### 사후검정 실시 (pg.pairwise_ttests)
import pingouin as pg

# H0 : 두 용량간 불안점수 평균은 같다.
# H1 : 두 용량간 불안점수 평균은 다르다.

result = pg.pairwise_ttests(df_melt,               ## 원본 데이터
                            dv="불안점수",         # 종속변수 컬럼명
                            within="용량",         # 독립변수 컬럼명
                            subject="참가자ID",    # 고유식별자 컬럼명
                            parametric=True,
                            padjust="bonf"         # 본페로니 보정
                            )
result

,Contrast,A,B,Paired,Parametric,T,dof,Tail,p-unc,p-corr,p-adjust,BF10,hedges
0,용량,저용량,중용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,2.674882
1,용량,저용량,고용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,5.349764
2,용량,중용량,고용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,2.674882


## <문제2>

반복측정 요인: 용량(저용량, 중용량, 고용량)
피험자간 요인: 성별(M/F)

다음을 검정하시오.

1. 용량에 따른 점수 차이가 유의한가? (반복측정 요인 주효과)

2. 성별에 따른 점수 차이가 유의한가? (피험자간 요인 주효과)

3. 성별과 용량 간 상호작용이 유의한가? (상호작용 효과)

4. 가설 설정을 작성하고, 반복측정 ANOVA를 수행한 후 결과를 해석하시오.

5. 사후검정이 필요하면 적절한 방법을 선택하여 수행하고 결과를 해석하시오.

In [33]:
file_path = "./data/repeated_anova2.csv"
df = pd.read_csv(file_path)
df

,참가자ID,성별,저용량,중용량,고용량
0,1,M,75,68,61
1,2,F,78,71,64
2,3,M,72,65,58
3,4,F,77,70,63
4,5,M,80,73,66
5,6,F,74,67,60
6,7,M,76,69,62
7,8,F,73,66,59
8,9,M,79,72,65
9,10,F,71,64,57


In [34]:
### 원본 데이터
df

,참가자ID,성별,저용량,중용량,고용량
0,1,M,75,68,61
1,2,F,78,71,64
2,3,M,72,65,58
3,4,F,77,70,63
4,5,M,80,73,66
5,6,F,74,67,60
6,7,M,76,69,62
7,8,F,73,66,59
8,9,M,79,72,65
9,10,F,71,64,57


In [35]:
### 그룹 나누기 (반복측정 요인별로만 그룹 묶기)
gLow = df["저용량"].to_numpy()
gMid = df["중용량"].to_numpy()
gHigh = df["고용량"].to_numpy()

In [36]:
### 정규성 검정 (gLow)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gLow)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.785484790802002
귀무가설 채택


In [37]:
### 정규성 검정 (gMid)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gMid)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.7854843735694885
귀무가설 채택


In [38]:
### 정규성 검정 (gHigh)
from scipy.stats import shapiro

# H0 : 데이터는 정규성을 만족한다.
# H1 : 데이터는 정규성을 만족하지 않는다.

s, p = shapiro(gHigh)

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")

검정통계량의 p-value : 0.7854846715927124
귀무가설 채택


In [39]:
### long 데이터로 변환
df_melt = pd.melt(df,
                  id_vars=["참가자ID", "성별"],
                  value_vars=["저용량", "중용량", "고용량"],
                  var_name="용량",
                  value_name="불안점수"
                  )
df_melt

,참가자ID,성별,용량,불안점수
0,1,M,저용량,75
1,2,F,저용량,78
2,3,M,저용량,72
3,4,F,저용량,77
4,5,M,저용량,80
5,6,F,저용량,74
6,7,M,저용량,76
7,8,F,저용량,73
8,9,M,저용량,79
9,10,F,저용량,71


In [40]:
### 구형성 검정(Mauchly's Test)
import pingouin as pg

# H0 : 데이터는 구형성을 만족한다.
# H1 : 데이터는 구형성을 만족하지 않는다.

result = pg.sphericity(df_melt,
                       dv="불안점수",
                       within="용량",      # 반복측정 요인 컬럼명만 적기
                       subject="참가자ID")
result

(True, inf, -inf, 2, 1.0)

In [64]:
### Mixed 반복측정 ANOVA 적합
import pingouin as pg

result = pg.mixed_anova(df_melt,             ## 원본 데이터
                        dv="불안점수",       # 종속변수 컬럼명
                        within="용량",       # 독립변수 컬럼명(반복측정요인 컬럼)
                        between="성별",      # 독립변수 컬럼명(집단간 요인 컬럼)
                        subject="참가자ID"   # 고유식별자 컬럼명
                        )
result

,Source,SS,DF1,DF2,MS,F,p-unc,np2,eps
0,성별,7.260000e+01,1,18,7.260000e+01,4.321429e+00,0.05221,0.1936,NaN
1,용량,1.960000e+03,2,36,9.800000e+02,-7.758154e+16,1.00000,1.0000,-0.0
2,Interaction,4.547474e-13,2,36,2.273737e-13,-1.800000e+01,1.00000,inf,NaN


In [65]:
### 용량에 따른 점수 차이가 유의한가? (반복측정 요인 주효과)

# H0 : 용량별 불안점수의 평균은 같다.
# H1 : 적어도 하나의 용량의 불안점수의 평균은 다르다.

p = result.loc[0, "p-unc"]

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")
print("유의수준 5%하에서 귀무가설을 채택한다. 즉, 용량별 불안점수의 평균은 같다.")

검정통계량의 p-value : 0.0522099741694285
귀무가설 채택
유의수준 5%하에서 귀무가설을 채택한다. 즉, 용량별 불안점수의 평균은 같다.


In [66]:
### 성별에 따른 점수 차이가 유의한가? (집단간 요인 주효과)

# H0 : 성별간 불안점수의 평균은 같다.
# H1 : 성별간 불안점수의 평균은 다르다.

p = result.loc[1, "p-unc"]

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")
print("유의수준 5%하에서 귀무가설을 채택한다. 즉, 성별간 불안점수의 평균은 같다.")

검정통계량의 p-value : 1.0
귀무가설 채택
유의수준 5%하에서 귀무가설을 채택한다. 즉, 성별간 불안점수의 평균은 같다.


In [68]:
### 성별과 용량 간 상호작용이 유의한가? (상호작용 효과)

# H0 : 용량과 성별간 상호작용 효과가 없다.
# H1 : 용량과 성별간 상호작용 효과가 있다.

p = result.loc[2, "p-unc"]

print(f"검정통계량의 p-value : {p}")
print("귀무가설 채택" if p > 0.05 else "귀무가설 기각")
print("유의수준 5%하에서 귀무가설을 채택한다. 즉, 용량과 성별간 상호작용 효과가 없다.")

검정통계량의 p-value : 1.0
귀무가설 채택
유의수준 5%하에서 귀무가설을 채택한다. 즉, 용량과 성별간 상호작용 효과가 없다.


In [69]:
print("""
귀무가설 채택이지만 귀무가설 기각일때에는 사후검정을 실시한다.
""")


귀무가설 채택이지만 귀무가설 기각일때에는 사후검정을 실시한다.



In [70]:
### (반복측정 요인) 사후 검정
import pingouin as pg

result = pg.pairwise_ttests(df_melt,
                            dv="불안점수",
                            within="용량",             # 반복측정 요인 컬럼명
                            subject="참가자ID",
                            padjust="bonf"
                            )
result

,Contrast,A,B,Paired,Parametric,T,dof,Tail,p-unc,p-corr,p-adjust,BF10,hedges
0,용량,저용량,중용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,2.674882
1,용량,저용량,고용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,5.349764
2,용량,중용량,고용량,True,True,inf,19.0,two-sided,0.0,0.0,bonf,nan,2.674882


In [71]:
### (집단 간 요인) 사후 검정
import pingouin as pg

result = pg.pairwise_ttests(df_melt,
                            dv="불안점수",
                            between="성별",        # 집단 간 요인 컬럼명
                            subject="참가자ID",
                            padjust="bonf"
                            )
result

,Contrast,A,B,Paired,Parametric,T,dof,Tail,p-unc,BF10,hedges
0,성별,M,F,False,True,1.364261,58.0,two-sided,0.177756,0.571,0.347676
